# Forecasting Baselines
This notebook evaluates our simple seasonal baseline models before generating more complex ML models. We can visualize the performance here.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Add src to module path for importing custom code
sys.path.append(os.path.abspath('../../src'))
from forecasting.baseline_models import (
    DailyNaiveForecaster, 
    WeeklyNaiveForecaster, 
    CombinedSeasonalForecaster, 
    evaluate_baselines
)

: 

In [ ]:
# Load data
# Note: As of the new integration interface, models need the entire raw data format 
# to anchor prediction bounds to specific solve_times over a 35h horizon.
from forecasting.data_cleaning import load_and_clean_data
from forecasting.fill_missing_data import fill_missing_linear

# Load full raw series
df_full = load_and_clean_data('../../data/forecasting/raw/raw_data_measured_demand.csv')
df_full = fill_missing_linear(df_full)

# Pick an arbitrary test split for evaluation / testing
solve_time = pd.Timestamp("2026-03-01T00:00:00Z")
horizon = 35

# history is everything up to the solve_time
df_history = df_full[df_full.index < solve_time]
# actuals is the exact 35 hour window we want to predict
y_test = df_full[(df_full.index >= solve_time) & (df_full.index < solve_time + pd.Timedelta(hours=horizon))]['heat_demand_W']

df_history.tail(3)

In [ ]:
# Dictionary to define all models to benchmark
models = {
    'Daily Naive': DailyNaiveForecaster(),
    'Weekly Naive': WeeklyNaiveForecaster(),
    'Combined Naive': CombinedSeasonalForecaster()
}

predictions = {}

# Generate predictions conforming to the new pipeline API
for name, model in models.items():
    # Note: ML models would typically have a model.fit(df_history) call here first
    
    # Models return predictions in Megawatts ('demand_mw_th'), we convert back to Watts 
    # to evaluate cleanly against 'heat_demand_W' backwards compatibility in this notebook
    pred_mw = model.predict(history=df_history, solve_time=solve_time, horizon=horizon)
    pred_w = pred_mw * 1e6
    predictions[name] = pred_w
    
    print(f"[{name}] Metrics:", evaluate_baselines(y_test, pred_w))

In [ ]:
# Visualizing the 35 hour prediction horizon side-by-side
# Since our test horizon is exactly 35 hours starting at solve_time
start = solve_time
end = solve_time + pd.Timedelta(hours=horizon)

plt.figure(figsize=(15, 6))
plt.plot(y_test.index, y_test.values, label='Actual Demand', color='black', linewidth=2)

for name, pred in predictions.items():
    plt.plot(pred.index, pred.values, label=name, alpha=0.7)

plt.title(f'Baseline Forecasts vs. Actual - 35 Hour Horizon from {start.strftime("%Y-%m-%d %H:%M")}')
plt.ylabel('Heat Demand (W)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Detailed Error Analysis

Let's compare the performance using our updated metrics:
- **MAE (Mean Absolute Error):** The average absolute difference between the forecast and actual demand. It tells us the expected error in Watts.
- **RMSE (Root Mean Squared Error):** Also in Watts, but squares the errors before averaging, effectively penalizing larger errors more heavily. Helpful for checking if the model misses big spikes.
- **MAPE (Mean Absolute Percentage Error):** The average relative percentage error. Useful for understanding the scale of the errors independent of the actual unit volume.
- **R² (Coefficient of Determination):** Measures the proportion of variance explained by the model. 1.0 represents a perfect forecast, 0.0 represents a model no better than just predicting the historical mean, and negative values indicate performance worse than the mean.

In [ ]:
import numpy as np

# Calculate comprehensive metrics dynamically
metrics_dict = {
    name: evaluate_baselines(y_test, pred)
    for name, pred in predictions.items()
}
metrics_df = pd.DataFrame(metrics_dict).T

# Formatting the output slightly better
display(metrics_df.round(2))

# Plot Metrics Comparison side-by-side
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# MAE & RMSE
metrics_df[['mae', 'rmse']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Absolute Error Metrics Comparison')
axes[0].set_ylabel('Heat Demand Error (W)')
axes[0].tick_params(axis='x', rotation=45)

# MAPE & R2
metrics_df[['mape_pct', 'r2']].plot(kind='bar', ax=axes[1])
axes[1].set_title('Relative/Scale-Independent Metrics Comparison')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Residuals Analysis for a specific model 
# (You can change 'best_model_name' to inspect any model in the dictionary)
best_model_name = 'Combined Naive' 

assert best_model_name in predictions, f"Model {best_model_name} not found in predictions dictionary"
residuals = y_test - predictions[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Histogram to evaluate residual normal distribution
axes[0].hist(residuals.dropna(), bins=60, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='dashed', linewidth=2)
axes[0].set_title(f'Residuals Distribution ({best_model_name})')
axes[0].set_xlabel('Error (Actual - Predicted) [W]')
axes[0].set_ylabel('Frequency')

# 2. Mean Absolute Error by Hour of Day
err_df = pd.DataFrame({'residual': residuals, 'hour': residuals.index.hour})
err_by_hour = err_df.groupby('hour')['residual'].apply(lambda x: np.mean(np.abs(x)))

axes[1].bar(err_by_hour.index, err_by_hour.values, color='lightcoral', alpha=0.8, edgecolor='black')
axes[1].set_title(f'Mean Abs Error by Hour ({best_model_name})')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('MAE [W]')
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()